# Laboratorio 2: motor de búsqueda semántica

Comparación entre embeddings multilingües y una búsqueda básica por palabras clave en un corpus de soporte para comercio electrónico.

In [ ]:
%pip install -q sentence-transformers numpy scikit-learn

## Configuración y datos

In [ ]:
import re
import unicodedata

import numpy as np
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
TOP_K = 3

CORPUS = [
    "No puedo iniciar sesión en mi cuenta.",
    "Olvidé mi contraseña y necesito recuperar el acceso.",
    "La cuenta fue bloqueada por demasiados intentos fallidos.",
    "Puedes restablecer tu clave desde la página de acceso.",
    "La verificación en dos pasos no envía el código de seguridad.",
    "Quiero actualizar el correo electrónico asociado a mi perfil.",
    "No recibí el mensaje para confirmar mi correo.",
    "La aplicación móvil no carga correctamente.",
    "La aplicación se cierra sola después de abrirla.",
    "El sitio web está temporalmente fuera de servicio.",
    "Revisa tu conexión a internet antes de volver a intentarlo.",
    "Debes actualizar la aplicación a la versión más reciente.",
    "El pedido ya fue enviado y está en camino.",
    "Puedes rastrear el paquete con el número de seguimiento.",
    "La entrega de mi compra está retrasada.",
    "Necesito modificar la dirección de entrega de mi pedido.",
    "Quiero devolver un producto porque llegó dañado.",
    "El reembolso puede tardar cinco días hábiles.",
    "Mi tarjeta fue rechazada al intentar pagar.",
    "Me cobraron dos veces por la misma compra.",
    "La factura se puede descargar desde el historial de pedidos.",
    "Necesito comunicarme con un agente de soporte.",
    "Las notificaciones de nuevos pedidos están desactivadas.",
    "Deseo eliminar definitivamente mi perfil de usuario.",
]

CONSULTAS = [
    "La aplicación móvil no carga correctamente",
    "Perdí mis credenciales y necesito volver a entrar",
    "La cuenta no funciona",
    "Deseo cambiar el email vinculado a mi usuario",
    "¿Dónde está mi compra?",
    "El pago de mi pedido aparece duplicado",
]

## Generación de embeddings

In [ ]:
modelo = SentenceTransformer(MODELO_EMBEDDINGS)
embeddings_corpus = modelo.encode(
    CORPUS,
    normalize_embeddings=True,
    convert_to_numpy=True,
)

print(f"Corpus: {len(CORPUS)} oraciones")
print(f"Dimensión de embeddings: {embeddings_corpus.shape[1]}")

## Búsqueda semántica y recuperación top-k

In [ ]:
def similitud_coseno(embedding_consulta, embeddings_indexados):
    return embeddings_indexados @ embedding_consulta


def buscar_semanticamente(consulta, corpus, embeddings_indexados, modelo, top_k=3):
    if not consulta.strip():
        raise ValueError("La consulta no puede estar vacía.")
    if not 1 <= top_k <= len(corpus):
        raise ValueError(f"top_k debe estar entre 1 y {len(corpus)}.")

    embedding_consulta = modelo.encode(
        consulta,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    puntajes = similitud_coseno(embedding_consulta, embeddings_indexados)
    indices = np.argsort(puntajes)[::-1][:top_k]

    return [
        {
            "rank": posicion,
            "indice": int(indice),
            "texto": corpus[indice],
            "score": float(puntajes[indice]),
        }
        for posicion, indice in enumerate(indices, start=1)
    ]

## Búsqueda por palabras clave

In [ ]:
def tokenizar_simple(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(caracter for caracter in texto if not unicodedata.combining(caracter))
    return set(re.findall(r"\b[a-z0-9]+\b", texto))


def buscar_por_palabras_clave(consulta, corpus, top_k=3):
    if not consulta.strip():
        raise ValueError("La consulta no puede estar vacía.")
    if not 1 <= top_k <= len(corpus):
        raise ValueError(f"top_k debe estar entre 1 y {len(corpus)}.")

    tokens_consulta = tokenizar_simple(consulta)
    resultados = []

    for indice, texto in enumerate(corpus):
        coincidencias = tokens_consulta & tokenizar_simple(texto)
        resultados.append({
            "indice": indice,
            "texto": texto,
            "score": len(coincidencias),
            "coincidencias": sorted(coincidencias),
        })

    resultados.sort(key=lambda resultado: (-resultado["score"], resultado["indice"]))
    for rank, resultado in enumerate(resultados[:top_k], start=1):
        resultado["rank"] = rank

    return resultados[:top_k]

## Comparación de resultados

In [ ]:
def imprimir_resultados(titulo, resultados):
    print(f"\n{titulo}")
    for resultado in resultados:
        if "coincidencias" in resultado:
            palabras = ", ".join(resultado["coincidencias"]) or "sin coincidencias"
            detalle = f"score={resultado['score']} | coincidencias={palabras}"
        else:
            detalle = f"score={resultado['score']:.4f}"
        print(
            f"  {resultado['rank']}. índice={resultado['indice']} | "
            f"{detalle} | {resultado['texto']}"
        )


def comparar_busquedas(consulta, top_k=TOP_K):
    semanticos = buscar_semanticamente(
        consulta, CORPUS, embeddings_corpus, modelo, top_k
    )
    palabras_clave = buscar_por_palabras_clave(consulta, CORPUS, top_k)

    print("=" * 100)
    print(f"CONSULTA: {consulta}")
    print("=" * 100)
    imprimir_resultados("Búsqueda semántica", semanticos)
    imprimir_resultados("Búsqueda por palabras clave", palabras_clave)

    return {"semantica": semanticos, "palabras_clave": palabras_clave}


def consultar(consulta, top_k=TOP_K):
    return comparar_busquedas(consulta, top_k)

In [ ]:
resultados_pruebas = {}

for consulta in CONSULTAS:
    resultados_pruebas[consulta] = comparar_busquedas(consulta)
    print()

## Probar una consulta propia

Cambia el texto y ejecuta la siguiente celda.

In [ ]:
consultar("necesito hablar con una persona", top_k=3);

## Reflexión requerida

> Pendiente: escribe aquí una reflexión de 150 a 250 palabras después de ejecutar y analizar las consultas.

Incluye cuál método funcionó mejor en cada caso relevante, un resultado inesperado, las limitaciones observadas y posibles mejoras al corpus o al método de recuperación.